# 합격자소서 임베딩 파이프라인 (차분 업데이트용)
**BGE-M3 → ChromaDB**  |  GPU 런타임 필수

### 이번 실행 목적
- 패턴② 재분리로 새로 생긴 qna ~154건만 추가 임베딩
- 기존 chroma_db(11,970벡터)에 이어서 작업

### 업로드할 파일 2개
- `essays.db` — 로컬 최신본 (qna_embeddings 11,915건 기록 포함)
- `chroma_db.zip` — 이전에 다운로드한 벡터 저장소

### 실행 순서
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. Cell 1~8 순서대로 실행

In [ ]:
# ── Cell 1: 패키지 설치 ─────────────────────────────────────
!pip install -q FlagEmbedding chromadb

In [ ]:
# ── Cell 2: GPU 확인 ────────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — 런타임을 GPU로 변경하세요')

In [ ]:
# ── Cell 3: 파일 업로드 (essays.db + chroma_db.zip) ─────────
from google.colab import files
import os, zipfile

print('essays.db 선택 → 업로드')
up1 = files.upload()
assert 'essays.db' in up1, 'essays.db 를 선택하세요'
print(f'essays.db: {os.path.getsize("essays.db"):,} bytes')

print('\nchroma_db.zip 선택 → 업로드')
up2 = files.upload()
assert 'chroma_db.zip' in up2, 'chroma_db.zip 을 선택하세요'
print(f'chroma_db.zip: {os.path.getsize("chroma_db.zip"):,} bytes')

# chroma_db.zip 압축 해제
with zipfile.ZipFile('chroma_db.zip', 'r') as z:
    z.extractall('/content/chroma_db')
print('chroma_db 압축 해제 완료')

In [ ]:
# ── Cell 4: 모델 로드 ───────────────────────────────────────
from FlagEmbedding import BGEM3FlagModel

MODEL_NAME = 'BAAI/bge-m3'
print('모델 로드 중...')
model = BGEM3FlagModel(MODEL_NAME, use_fp16=True)
print('모델 로드 완료')

In [ ]:
# ── Cell 5: ChromaDB 연결 ────────────────────────────────────
import chromadb

client     = chromadb.PersistentClient(path='/content/chroma_db')
collection = client.get_or_create_collection(
    name='cover_letters',
    metadata={'hnsw:space': 'cosine'},
)
print(f'ChromaDB 준비 | 기존 벡터: {collection.count()}개')  # 약 11,970개 보여야 정상

In [ ]:
# ── Cell 6: 미임베딩 QnA 로드 ───────────────────────────────
import sqlite3

conn = sqlite3.connect('essays.db')

COLS = ['qna_id','essay_id','question_clean','answer_clean','question_type',
        'char_count','source','company','org_type','role','hire_type',
        'year','season','university','major']

rows = conn.execute("""
    SELECT
        q.id, q.essay_id,
        q.question_clean, q.answer_clean,
        q.question_type, q.char_count,
        e.source, e.company, e.org_type,
        e.role, e.hire_type, e.year, e.season,
        e.university, e.major
    FROM  qna q
    JOIN  essays e ON e.id = q.essay_id
    LEFT JOIN qna_embeddings qe ON qe.qna_id = q.id
    WHERE q.is_valid = 1
      AND q.question_clean IS NOT NULL
      AND qe.id IS NULL
    ORDER BY q.id
""").fetchall()

pending = [dict(zip(COLS, r)) for r in rows]
print(f'임베딩 대상: {len(pending)}건')  # 약 154건 보여야 정상

In [ ]:
# ── Cell 7: 임베딩 실행 ─────────────────────────────────────
import time, sqlite3
from datetime import datetime
from tqdm.auto import tqdm

conn = sqlite3.connect('essays.db')

BATCH_SIZE = 64

def make_text(q, a):
    q = (q or '').strip()
    a = (a or '').strip()
    return f'{q}\n{a}' if q else a

def mark_done(conn, qna_ids, chroma_ids):
    now = datetime.now().isoformat()
    conn.executemany(
        'INSERT OR IGNORE INTO qna_embeddings (qna_id, chroma_id, model_name, embedded_at) VALUES (?,?,?,?)',
        [(qid, cid, MODEL_NAME, now) for qid, cid in zip(qna_ids, chroma_ids)],
    )
    conn.commit()

total = len(pending)
done  = failed = 0
t0    = time.time()

for start in tqdm(range(0, total, BATCH_SIZE), desc='임베딩'):
    batch = pending[start : start + BATCH_SIZE]
    texts = [make_text(r['question_clean'], r['answer_clean']) for r in batch]

    try:
        out  = model.encode(texts, batch_size=BATCH_SIZE, max_length=8192)
        vecs = out['dense_vecs']
    except Exception as e:
        print(f'임베딩 오류 (idx={start}): {e}')
        failed += len(batch)
        continue

    chroma_ids = [f"qna_{r['qna_id']}" for r in batch]
    qna_ids    = [r['qna_id'] for r in batch]
    metadatas  = [
        {
            'qna_id':        int(r['qna_id']),
            'essay_id':      int(r['essay_id']),
            'source':        r['source']        or '',
            'company':       r['company']        or '',
            'org_type':      r['org_type']       or '',
            'role':          r['role']           or '',
            'hire_type':     r['hire_type']      or '',
            'year':          r['year']           or '',
            'season':        r['season']         or '',
            'question_type': r['question_type']  or '',
            'university':    r['university']     or '',
            'major':         r['major']          or '',
            'char_count':    int(r['char_count'] or 0),
        }
        for r in batch
    ]

    try:
        collection.upsert(
            ids=chroma_ids,
            embeddings=vecs.tolist(),
            metadatas=metadatas,
            documents=texts,
        )
        mark_done(conn, qna_ids, chroma_ids)
        done += len(batch)
    except Exception as e:
        print(f'upsert 오류 (idx={start}): {e}')
        failed += len(batch)

elapsed = time.time() - t0
print(f'\n완료: 성공={done}건  실패={failed}건  소요={elapsed:.1f}초  ({elapsed/60:.1f}분)')
print(f'ChromaDB 총 벡터: {collection.count()}개')

In [ ]:
# ── Cell 8: 결과 다운로드 ────────────────────────────────────
import shutil, os
conn.close()

shutil.make_archive('/content/chroma_db', 'zip', '/content/chroma_db')
print(f'chroma_db.zip: {os.path.getsize("/content/chroma_db.zip"):,} bytes')

files.download('/content/chroma_db.zip')  # 벡터 저장소
files.download('/content/essays.db')      # qna_embeddings 완전히 채워진 DB